# 04 — Build and search the local lexical index

BM25 is OSII's zero-model retrieval baseline. It creates overlapping,
provenance-aware chunks from preferred text, then ranks exact and related
word matches. No model service, network access, or container is required.

In [1]:
from osii.domain.scopes.collections import list_collections
from osii.domain.services.search import dashboard_search
from osii.search.lexical import build_bm25_index

from _demo_support import demo_paths, heading, require_path


paths = demo_paths()
require_path(paths.osii_root / "objects", "Run the earlier numbered examples first.")

index_path, metadata_path = build_bm25_index(paths.osii_root)

heading("Index files")
print("BM25 index:", index_path)
print("Metadata:", metadata_path)

/Users/heidi/Projects/OSII/osii/osii-demo-notebooks/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                                                                       


Index files
-----------
BM25 index: /Users/heidi/Projects/OSII/osii/osii-demo-notebooks/demo-workspace/.osii/embeddings/lexical/bm25_index.pkl
Metadata: /Users/heidi/Projects/OSII/osii/osii-demo-notebooks/demo-workspace/.osii/embeddings/lexical/bm25_meta.json


## Search the complete library

Search results retain object, segment, page when available, and character
offsets so the dashboard or an agent can return to the evidence.

In [2]:
mode_used, results = dashboard_search(
    paths.osii_root,
    query="low Reynolds number viscosity swimming microorganisms",
    mode="lexical",
    top_k=5,
    scope={"scope_type": "root"},
)

heading(f"Root results ({mode_used})")
for result in results:
    print(f"- {result['source_relpath']}  score={result['score']:.3f}")
    print(f"  {result['snippet']}")
    print(f"  chars={result.get('char_start')}:{result.get('char_end')}")


Root results (lexical)
----------------------
- purcell.pdf  score=63.000
  The demonstration involved a tall rectangular transparent vessel of corn syrup, projected by an overhead projector turned on its side. Some essential hand waving could not be reproduced.

This is a talk that I would not, I’m afraid, have the nerve to give under any other circumstances, It’s a story I’ve been saving up to tell Viki. Like so many of you here, I’ve enjoyed from time to time the wonde
  chars=550:1279
- purcell.pdf  score=42.000
  We wander around strictly as amateurs equipped only with some elementary physics, and in the end, it turns out, we improve our understanding of the elemen- tary physics even if we don’t throw much light on the other subjects. Now this is that kind of a subject, but I have still another reason for wanting to, as it were, needle Viki with it, because I’m going to talk for a while about viscosity. Vi
  chars=1054:1808
- purcell.pdf  score=10.000
  The viscosity of a liquid 

## Run the same search inside a collection

In [3]:
collection = next(
    item for item in list_collections(paths.osii_root) if item["name"] == "Purcell analysis"
)
_, collection_results = dashboard_search(
    paths.osii_root,
    query="scallop theorem reciprocal motion",
    mode="lexical",
    top_k=5,
    scope={"scope_type": "collection", "collection_id": collection["id"]},
)

heading("Collection-scoped results")
for result in collection_results:
    print("-", result["source_relpath"], "->", result["snippet"])


Collection-scoped results
-------------------------
- purcell.pdf -> Life at low Reynolds number

E. M. Purcell Lyman Laboratory, Harvard University, Cambridge, Massachusetts 02138 (Received 12 June 1976)

Editor’s note: This is a reprint (slightly edited) of a paper of the same title that appeared in the book Physics and Our World: A Symposium in Honor of Victor F. Weisskopf, published by the American Institute of Physics (1976). The personal tone of the original
- purcell.pdf -> The demonstration involved a tall rectangular transparent vessel of corn syrup, projected by an overhead projector turned on its side. Some essential hand waving could not be reproduced.

This is a talk that I would not, I’m afraid, have the nerve to give under any other circumstances, It’s a story I’ve been saving up to tell Viki. Like so many of you here, I’ve enjoyed from time to time the wonde
- purcell.pdf -> We wander around strictly as amateurs equipped only with some elementary physics, and in the en

Lexical retrieval remains available even if every optional embedding or LLM
endpoint is offline. The next script adds the deterministic hashing-vector
baseline without claiming that those vectors are semantic.